In [0]:
%sql
CREATE VOLUME workspace.default.retail_volume;

In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/retail_volume/retail_db/customers/part-00000",
    header=False,
    inferSchema=True
)

display(df)

Error in callback <bound method UserNamespaceCommandHook.post_run_cell of <dbruntime.DatasetInfo.UserNamespaceCommandHook object at 0xff4f049d8c80>> (for post_run_cell), with arguments args (<ExecutionResult object at ff4efde456d0, execution_count=2 error_before_exec=None error_in_exec= info=<ExecutionInfo object at ff4f045a5790, raw_cell="df = spark.read.csv(
    "/Volumes/workspace/defau.." transformed_cell="df = spark.read.csv(
    "/Volumes/workspace/defau.." store_history=True silent=False shell_futures=True cell_id=8247185119084143> result=None>,),kwargs {}:


In [0]:
%fs ls dbfs:/

path,name,size,modificationTime
dbfs:/Volumes/,Volumes/,0,0
dbfs:/Workspace/,Workspace/,0,0
dbfs:/databricks-datasets/,databricks-datasets/,0,0


In [0]:
schemas_text = spark.read.text('/Volumes/workspace/default/retail_volume/retail_db/schemas.json',wholetext = True).first().value


In [0]:
import json
column_details = json.loads(schemas_text)['orders']

In [0]:
col_sorted = [col['column_name'] for col in sorted(column_details, key=lambda col: col['column_position'])]

In [0]:
order_path = '/Volumes/workspace/default/retail_volume/retail_db/orders/part-00000'

In [0]:
orders = spark.read.csv(order_path, inferSchema=True).toDF(*col_sorted)

In [0]:
from pyspark.sql.functions import count, col

In [0]:
orders.groupBy('order_status').agg(count('*').alias('order_count')).orderBy(col('order_count').desc()).show()

+---------------+-----------+
|   order_status|order_count|
+---------------+-----------+
|       COMPLETE|      22899|
|PENDING_PAYMENT|      15030|
|     PROCESSING|       8275|
|        PENDING|       7610|
|         CLOSED|       7556|
|        ON_HOLD|       3798|
|SUSPECTED_FRAUD|       1558|
|       CANCELED|       1428|
| PAYMENT_REVIEW|        729|
+---------------+-----------+



In [0]:
schemas_path = '/Volumes/workspace/default/retail_volume/retail_db/schemas.json'

In [0]:
def get_column_names(schemas_path, ds_name):
    schemas_text = spark.read.text(schemas_path ,wholetext = True).first().value
    schemas = json.loads(schemas_text)
    column_details = schemas[ds_name]
    col_sorted = [col['column_name'] for col in sorted(column_details, key=lambda col: col['column_position'])]
    return col_sorted

In [0]:
get_column_names(schemas_path, 'customers')

['customer_id',
 'customer_fname',
 'customer_lname',
 'customer_email',
 'customer_password',
 'customer_street',
 'customer_city',
 'customer_state',
 'customer_zipcode']

In [0]:
ds_list = ['customers','orders','products','departments','order_items','categories']

In [0]:
base_dir = '/Volumes/workspace/default/retail_volume/retail_db'

In [0]:
import os
tgt_base_dir = '/Volumes/workspace/default/retail_volume/tgt_retail_db'
os.makedirs(tgt_base_dir, exist_ok=True)

In [0]:
for ds in ds_list:
    print(f'processing {ds} data')
    columns = get_column_names(f'{base_dir}/schemas.json',ds)
    df = spark.read.csv(f'{base_dir}/{ds}', inferSchema=True).toDF(*columns)
    df.write.mode('overwrite').parquet(f'{tgt_base_dir}/{ds}')

processing customers data
processing orders data
processing products data
processing departments data
processing order_items data
processing categories data
